In [1]:
import os
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


2025-11-08 07:17:57.981411: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-08 07:17:58.031497: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-08 07:17:59.327143: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ----------------------------
# Set up dataset paths
# ----------------------------

DATA_DIR = "binary_model_data"     # Main dataset folder
TRAIN_DIR = os.path.join(DATA_DIR, "train")   # Training data
VAL_DIR = os.path.join(DATA_DIR, "valid")     # Validation data
TEST_DIR = os.path.join(DATA_DIR, "test")     # Test data

In [3]:
# ----------------------------
# Create output folder
# ----------------------------

OUTPUT_DIR = "./efficientnetb0_base"     # Folder to save model outputs
os.makedirs(OUTPUT_DIR, exist_ok=True) # Make directory if it doesn't exist

In [4]:
# ----------------------------
# Training parameters
# ----------------------------

IMAGE_SIZE = (224, 224)     # Input image size
BATCH_SIZE = 32             # Number of images per batch
SEED = 42                   # Random seed for reproducibility
EPOCHS = 8                  # Number of training epochs
LEARNING_RATE = 1e-4        # Learning rate for optimizer
USE_MIXED_PRECISION = True  # Enable mixed precision training

In [5]:
# ----------------------------
# Mixed precision for speed
# ----------------------------
if USE_MIXED_PRECISION:
    try:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision enabled.")
    except Exception as e:
        print("Mixed precision not available:", e)

Mixed precision enabled.


In [6]:
# ----------------------------
# Load datasets (train/valid/test)
# ----------------------------

AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(   # Load training data
    TRAIN_DIR,
    label_mode="binary",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(        # Load validation data
    VAL_DIR,
    label_mode="binary",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(        # Load test data
    TEST_DIR,
    label_mode="binary",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

print("Class names:", train_ds.class_names) # Show class labels found in training set

Found 12383 files belonging to 2 classes.
Found 991 files belonging to 2 classes.
Found 603 files belonging to 2 classes.
Class names: ['not_skin', 'skin']


2025-11-08 07:19:37.632847: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
# ----------------------------
# Data augmentation & preprocessing
# ----------------------------

# Apply random transformations to make the model more robust
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),      # Flip images horizontally
    layers.RandomRotation(0.06),          # Slightly rotate images
    layers.RandomZoom(0.08),              # Random zoom in/out
    layers.RandomTranslation(0.04, 0.04), # Shift images in width and height
    layers.RandomContrast(0.08)           # Adjust image contrast
])

In [9]:
# ----------------------------
# Preprocessing for EfficientNetB0
# ----------------------------

from tensorflow.keras.applications.efficientnet import preprocess_input

# For training data: apply augmentation + EfficientNet preprocessing
def preprocess_train(images, labels):
    images = tf.cast(images, tf.float32)   # Convert to float32
    images = data_augmentation(images)     # Apply random augmentations
    images = preprocess_input(images)      # Normalize for EfficientNet
    return images, labels

# For validation/test data: only preprocess, no augmentation
def preprocess_eval(images, labels):
    images = tf.cast(images, tf.float32)   # Convert to float32
    images = preprocess_input(images)      # Normalize for EfficientNet
    return images, labels

In [10]:
# ----------------------------
# Prepare datasets for training
# ----------------------------

# Apply preprocessing, cache for faster access, and prefetch for efficient loading
train_ds = train_ds.map(preprocess_train, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_ds = val_ds.map(preprocess_eval, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds = test_ds.map(preprocess_eval, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)


In [11]:
# ----------------------------
# Build EfficientNetB0 base model
# ----------------------------

from tensorflow.keras.applications import EfficientNetB0

# Load EfficientNetB0 with pretrained ImageNet weights (without top layers)
base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base model layers

# Build the full model
inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)     # Convert feature maps to a single vector
x = layers.Dropout(0.3)(x)                 # Dropout for regularization
outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)  # Binary classification output

model = keras.Model(inputs, outputs, name="efficientnetb0_base")
model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "efficientnetb0_base"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [12]:
# ----------------------------
# Compile and train
# ----------------------------
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[
        "accuracy",
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(),
        keras.metrics.Recall()
    ]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "best_base.keras"),
        save_best_only=True,
        monitor="val_loss"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3
    )
]

print("Training EfficientNetB0 base model...")
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

Training EfficientNetB0 base model...
Epoch 1/8
387/387 ━━━━━━━━━━━━━━━━━━━━ 396s 998ms/step - accuracy: 0.8273 - auc: 0.9210 - loss: 0.4341 - precision: 0.7954 - recall: 0.8821 - val_accuracy: 0.9425 - val_auc: 0.9933 - val_loss: 0.2538 - val_precision: 0.9028 - val_recall: 0.9919 - learning_rate: 1.0000e-04
Epoch 2/8
387/387 ━━━━━━━━━━━━━━━━━━━━ 286s 740ms/step - accuracy: 0.9464 - auc: 0.9894 - loss: 0.2187 - precision: 0.9226 - recall: 0.9747 - val_accuracy: 0.9637 - val_auc: 0.9968 - val_loss: 0.1699 - val_precision: 0.9356 - val_recall: 0.9960 - learning_rate: 1.0000e-04
Epoch 3/8
387/387 ━━━━━━━━━━━━━━━━━━━━ 271s 702ms/step - accuracy: 0.9589 - auc: 0.9935 - loss: 0.1588 - precision: 0.9396 - recall: 0.9810 - val_accuracy: 0.9667 - val_auc: 0.9977 - val_loss: 0.1365 - val_precision: 0.9393 - val_recall: 0.9980 - learning_rate: 1.0000e-04
Epoch 4/8
387/387 ━━━━━━━━━━━━━━━━━━━━ 285s 738ms/step - accuracy: 0.9678 - auc: 0.9957 - loss: 0.1268 - precision: 0.9519 - recall: 0.9855 - v

In [13]:
# ----------------------------
# Save model
# ----------------------------

base_model_path = os.path.join(OUTPUT_DIR, "efficientnetb0_base.keras")
model.save(base_model_path)
print("Base model saved at:", base_model_path)

Base model saved at: ./efficientnetb0_base/efficientnetb0_base.keras


In [14]:
# ----------------------------
# Evaluate on test set
# ----------------------------

print("Evaluating on test set:\n")

# Evaluate model on test data
results = model.evaluate(test_ds, return_dict=True, verbose=0)

print("Evaluation Results:")
for name, value in results.items():
    print(f"{name.capitalize():<10}: {value:.4f}")

Evaluating on test set:

Evaluation Results:
Accuracy  : 0.9735
Auc       : 0.9980
Loss      : 0.0973
Precision : 0.9554
Recall    : 0.9934
